# Compare three director-color schemes

This notebook compares the same three director-to-color maps in two ways:

1. on the director color sphere itself;
2. on the bundled `Q_example_workflow.npy` field, using the same practical visualization recipe as `tutorials/quick_visualize_q.ipynb`.

The three schemes are:

- original Nematics3D `n_color_immerse()`;
- previous Stage-I sRGB Pareto candidate, `director_color_pareto_034`;
- selected OKLab Stage-I Pareto knee, `director_color_pareto_oklab_043`.

For each comparison, geometry and visualization parameters are kept fixed. Only the director color function changes.

In [ ]:
from pathlib import Path
import sys

import numpy as np

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "example" / "data" / "Q_example_workflow.npy").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root from the current working directory."
    )

REPO_ROOT = find_repo_root()

try:
    import nematics3d as n3d
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import nematics3d as n3d

from nematics3d.field import n_color_immerse
from nematics3d.classes.q_field_object import QFieldObject
from nematics3d.classes.visual.plot_figure import PlotFigure
from nematics3d.classes.visual.plot_tube import OptsTube
from nematics3d.classes.visual.color import (
    director_color_pareto_034,
    director_color_pareto_oklab_043,
    plot_director_color_sphere,
)
from nematics3d.quick import (
    _auto_quick_Q_visual_params,
    _resolve_director_spacing_level,
)

DATA_PATH = REPO_ROOT / "example" / "data" / "Q_example_workflow.npy"
Q_data = np.load(DATA_PATH)
Q_data.shape

## Part I — color spheres

These three plots show the raw geometry of the three color maps on the director sphere.

### 1. Original Nematics3D scheme

In [ ]:
scene_nematics3d = plot_director_color_sphere(
    n_color_immerse,
    figure_size=(1000, 1000),
)
scene_nematics3d

### 2. Previous sRGB Pareto candidate ($J_{\mathrm{norm}}=0.34$)

In [ ]:
scene_srgb_pareto = plot_director_color_sphere(
    director_color_pareto_034,
    figure_size=(1000, 1000),
)
scene_srgb_pareto

### 3. OKLab Pareto knee ($J_{\mathrm{loc}}^{\mathrm{OKLab}}\approx0.43$)

In [ ]:
scene_oklab_pareto = plot_director_color_sphere(
    director_color_pareto_oklab_043,
    figure_size=(1000, 1000),
)
scene_oklab_pareto

## Part II — practical Q-field comparison

Now use the same bundled Q-tensor data and essentially the same construction as `quick_visualize_q()`.

To make the comparison clean, the Q field is initialized and its disclination lines are smoothed only once. Each of the three figures then receives the same line geometry, box extent, director-plane position, spacing, rod length, and rod radius. The only changed parameter is `n_color`.

The director plane uses the same default `grid_normal=(0,0,1)` and `director_spacing="medium"` as `quick_visualize_q()`.

In [ ]:
grid_normal = (0, 0, 1)
director_spacing = "medium"

params = _auto_quick_Q_visual_params(Q_data, grid_normal)
director_spacing_config = _resolve_director_spacing_level(director_spacing)

Q_obj = QFieldObject(
    Q=Q_data,
    name="director-color-comparison",
    default_miminum_line_length_smooth=params["smooth_min_line_length"],
    default_smooth_window_length=params["smooth_window_length"],
    default_miminum_line_length_visual=params["visual_min_line_length"],
)

Q_obj.act_lines_smooth(
    min_line_length=params["smooth_min_line_length"],
    window_length=params["smooth_window_length"],
)

In [ ]:
def make_qfield_color_comparison_figure(color_func, label):
    figure = PlotFigure()

    Q_obj.act_visualize_disclination_lines(
        figure=figure,
        is_extent=False,
        min_line_length=params["visual_min_line_length"],
        line_radius=params["line_radius"],
    )

    Q_obj.calc_bounds.act_visualize(
        figure=figure,
        opts=OptsTube(radius=params["extent_radius"]),
        is_reset_camera=False,
    )

    Q_obj.act_visualize_n_plane(
        figure=figure,
        is_extent=False,
        grid_normal=grid_normal,
        grid_spacing=(
            params["grid_spacing"]
            * director_spacing_config["grid_spacing_scale"]
        ),
        grid_size=params["grid_size"],
        grid_origin=params["grid_origin"],
        n_length=(
            params["n_length"]
            * director_spacing_config["n_length_scale"]
        ),
        n_radius=(
            params["n_radius"]
            * director_spacing_config["n_radius_scale"]
        ),
        n_color=color_func,
        plane_name=f"n-plane-{label}",
    )

    return figure

### 1. Original Nematics3D colors on the Q field

In [ ]:
figure_q_original = make_qfield_color_comparison_figure(
    n_color_immerse,
    "original",
)
figure_q_original

### 2. Previous sRGB Pareto colors on the Q field

In [ ]:
figure_q_srgb = make_qfield_color_comparison_figure(
    director_color_pareto_034,
    "srgb-pareto-034",
)
figure_q_srgb

### 3. OKLab Pareto-knee colors on the Q field

In [ ]:
figure_q_oklab = make_qfield_color_comparison_figure(
    director_color_pareto_oklab_043,
    "oklab-pareto-043",
)
figure_q_oklab

The second set of three figures is the practical comparison to inspect most closely. Since the Q field, camera construction, defects, plane geometry, and rod geometry are held fixed, visible differences among these figures come from the director color mapping rather than from a changed dataset or sampling choice.